In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT FILES
# =============================================================================

HAZARD_CSV = Path("data") / "hazard.csv"
EXPOSURE_CSV = Path("data") / "exposure_district.csv"
VULNERABILITY_CSV = Path("data") / "vulnerability.csv"
GOVT_CSV = Path("data") / "government_response_district.csv"
MASTER_CSV = Path("data") / "MASTER_VARIABLES.csv"

hazard = pd.read_csv(HAZARD_CSV)
exposure = pd.read_csv(EXPOSURE_CSV)
vulnerability = pd.read_csv(VULNERABILITY_CSV)
gov = pd.read_csv(GOVT_CSV)

# =============================================================================
# STANDARDIZE COLUMN NAMES
# (Replace "_" with "-" only in headers)
# =============================================================================

hazard.columns = hazard.columns.str.replace("_", "-", regex=False)
exposure.columns = exposure.columns.str.replace("_", "-", regex=False)
vulnerability.columns = vulnerability.columns.str.replace("_", "-", regex=False)
gov.columns = gov.columns.str.replace("_", "-", regex=False)

# =============================================================================
# KEEP REQUIRED COLUMNS
# =============================================================================

hazard = hazard[["district", "timeperiod", "heat-hazard", "heat-days-score"]]
exposure = exposure[["district", "timeperiod", "exposure"]]
vulnerability = vulnerability[["district", "timeperiod", "vulnerability"]]
gov = gov[["district", "timeperiod", "government-response"]]

# =============================================================================
# MERGE COMPONENTS (DISTRICT × TIMEPERIOD)
# =============================================================================

df = hazard.merge(exposure, on=["district", "timeperiod"], how="inner")
df = df.merge(vulnerability, on=["district", "timeperiod"], how="inner")
df = df.merge(gov, on=["district", "timeperiod"], how="inner")

# =============================================================================
# STANDARDIZE MERGE KEYS
# =============================================================================

df["district"] = df["district"].astype(str).str.strip()
df["timeperiod"] = df["timeperiod"].astype(str).str.strip()

# =============================================================================
# WEIGHTS
# =============================================================================

weights = {
    "heat-hazard": 4,
    "exposure": 1,
    "vulnerability": 2,
    "government-response": 2,
}

total_weight = sum(weights.values())

# =============================================================================
# TOPSIS PER TIMEPERIOD
# =============================================================================

results = []

for tp, g in df.groupby("timeperiod"):
    g = g.copy()

    # -------------------------------------------------------------------------
    # Min-Max Normalization
    # -------------------------------------------------------------------------
    norm = pd.DataFrame(index=g.index)

    for col in weights:
        min_v = g[col].min()
        max_v = g[col].max()

        if max_v == min_v:
            norm[col] = 0
        else:
            norm[col] = (g[col] - min_v) / (max_v - min_v)

    # -------------------------------------------------------------------------
    # Apply weights
    # -------------------------------------------------------------------------
    for col in weights:
        norm[col] *= weights[col] / total_weight

    # -------------------------------------------------------------------------
    # Ideal Best / Worst
    # -------------------------------------------------------------------------
    ideal_best = norm.max()
    ideal_worst = norm.min()

    # -------------------------------------------------------------------------
    # Euclidean Distances
    # -------------------------------------------------------------------------
    dist_best = np.sqrt(((norm - ideal_best) ** 2).sum(axis=1))
    dist_worst = np.sqrt(((norm - ideal_worst) ** 2).sum(axis=1))

    # -------------------------------------------------------------------------
    # TOPSIS Score
    # -------------------------------------------------------------------------
    g["topsis-score"] = dist_worst / (dist_best + dist_worst)

    results.append(g)

# =============================================================================
# COMBINE RESULTS
# =============================================================================

df = pd.concat(results, ignore_index=True)

# =============================================================================
# RISK CLASSIFICATION
# =============================================================================

def classify(score):
    if score <= 0.2:
        return 1
    elif score <= 0.4:
        return 2
    elif score <= 0.6:
        return 3
    elif score <= 0.8:
        return 4
    else:
        return 5


df["heat-risk-score"] = df["topsis-score"].apply(classify)

# =============================================================================
# SUMMARY
# =============================================================================

print("\nTOPSIS Summary:")
print(df["topsis-score"].describe())

print("\nRisk Class Distribution:")
print(df["heat-risk-score"].value_counts().sort_index())

print("\nPreview:")
print(df[["district", "timeperiod", "topsis-score", "heat-risk-score"]].head())

# =============================================================================
# APPEND RESULTS TO MASTER_VARIABLES
# =============================================================================

master = pd.read_csv(MASTER_CSV)

# Standardize headers in MASTER_VARIABLES too
master.columns = master.columns.str.replace("_", "-", regex=False)

master["district"] = master["district"].astype(str).str.strip()
master["timeperiod"] = master["timeperiod"].astype(str).str.strip()

# Remove old columns if present
for col in [
    "heat-hazard",
    "exposure",
    "vulnerability",
    "government-response",
    "topsis-score",
    "heat-risk-score",
]:
    if col in master.columns:
        master = master.drop(columns=col)

# Merge scores back
final_df = master.merge(
    df[
        [
            "district",
            "timeperiod",
            "heat-hazard",
            "exposure",
            "vulnerability",
            "government-response",
            "topsis-score",
            "heat-risk-score",
        ]
    ],
    on=["district", "timeperiod"],
    how="left",
)

# =============================================================================
# SAVE FINAL OUTPUT
# =============================================================================

OUTPUT = Path("data") / "final_risk_score.csv"

final_df.to_csv(OUTPUT, index=False)

print(f"\nSaved final file: {OUTPUT}")
print(f"Rows: {len(final_df)}")
print(f"Columns: {len(final_df.columns)}")


# =============================================================================
# DISTRICT-LEVEL FINAL RISK SCORE
# =============================================================================

district_df = (
    final_df.groupby(["district", "timeperiod"], as_index=False)
    .agg(
        {
            # Identifiers
            "dtname": "first",

            # Sum
            "HealthCenters": "sum",

            # Means
            "total-tender-awarded-value": "mean",
            "mean-heatday": "mean",
            "sum-aged-population": "mean",
            "sum-young-population": "mean",
            "sum-population": "mean",
            "avg-electricity": "mean",
            "block-piped-hhds-pct": "mean",
            "block-nosanitation-hhds-pct": "mean",
            "total-hhd": "mean",
            "nco-5-9-percent-estimated": "mean",
            "women-sugar": "mean",
            "men-sugar": "mean",
            "women-bp": "mean",
            "men-bp": "mean",
            "pct-ncd": "mean",
            "land-surface-temperature-raster": "first",
            "heat-hazard": "mean",
            "exposure": "mean",
            "vulnerability": "mean",
            "government-response": "mean",
            "topsis-score": "mean",
            "heat-risk-score": "mean",
        }
    )
)

# Round numeric columns
numeric_cols = district_df.select_dtypes(include=np.number).columns
district_df[numeric_cols] = district_df[numeric_cols].round(3)

# Create district -> object_id lookup
master_df = pd.read_csv(MASTER_CSV)
object_lookup = (
    master_df[["district", "object_id"]]
    .drop_duplicates(subset="district")
)
# Add object_id to district_df
district_df = district_df.merge(
    object_lookup,
    on="district",
    how="left"
)

# Keep only the first two parts of the object_id to represent only the district
district_df["object_id"] = (
    district_df["object_id"]
    .astype(str)
    .str.rsplit("-", n=1)
    .str[0]
)

# Rename columns
district_df = district_df.rename(
    columns={
        "HealthCenters": "health-centres-count",
        "block-piped-hhds-pct": "piped-hhds-pct",
        "block-nosanitation-hhds-pct": "nosanitation-hhds-pct",
        "mean-heatday": "heat-days-score",
        "nco-5-9-percent-estimated": "workers-affected-pct",
    }
)

# =============================================================================
# SAVE
# =============================================================================


DISTRICT_OUTPUT = Path("data") / "district_final_risk_score.csv"

district_df.to_csv(DISTRICT_OUTPUT, index=False)

print(f"\nSaved district-level file: {DISTRICT_OUTPUT}")
print(f"Rows: {len(district_df)}")
print(f"Columns: {len(district_df.columns)}")

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

DISTRICT_OUTPUT = Path("data") / "district_final_risk_score.csv"

district_df.to_csv(DISTRICT_OUTPUT, index=False)

print(f"\nSaved district-level file: {DISTRICT_OUTPUT}")
print(f"Rows: {len(district_df)}")
print(f"Columns: {len(district_df.columns)}")


TOPSIS Summary:
count    1890.000000
mean        0.451354
std         0.201774
min         0.000000
25%         0.314187
50%         0.485495
75%         0.514505
max         1.000000
Name: topsis-score, dtype: float64

Risk Class Distribution:
heat-risk-score
1    289
2    351
3    907
4    266
5     77
Name: count, dtype: int64

Preview:
    district timeperiod  topsis-score  heat-risk-score
0     Anugul    2021_01      0.271192                2
1   Balangir    2021_01      0.195194                1
2  Baleshwar    2021_01      0.372155                2
3    Bargarh    2021_01      0.104778                1
4    Bhadrak    2021_01      0.585786                3

Saved final file: data/final_risk_score.csv
Rows: 19782
Columns: 32

Saved district-level file: data/district_final_risk_score.csv
Rows: 1890
Columns: 27

Saved district-level file: data/district_final_risk_score.csv
Rows: 1890
Columns: 27


In [11]:
from pathlib import Path

print("CWD:", Path.cwd())
print("Hazard exists:", Path("data/hazard.csv").exists())
print("Exposure exists:", Path("data/exposure.csv").exists())
print("Vulnerability exists:", Path("data/vulnerability.csv").exists())

CWD: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel
Hazard exists: True
Exposure exists: True
Vulnerability exists: True
